# Testnotebook for Evaluation

In [ ]:
import anndata

#h5ad_file='../data/blood/10x-rep1-kallisto-cellbender/10x-rep1-kallisto-cellbender'
#h5ad_file='../data/blood/10x-rep2-kallisto-cellbender/10x-rep2-kallisto-cellbender'
#h5ad_file='../data/blood/bd-rhap-rep1/bd-rhap-rep1'
#h5ad_file='../data/blood/bd-rhap-rep2/bd-rhap-rep2'
h5ad_file='../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design'

# Read data from h5ad files
adata = anndata.io.read_h5ad(f'{h5ad_file}.h5ad')
print(adata)

In [1]:
import scanpy as sc
import decoupler as dc
import pandas as pd
import anndata as ad

# 1. Load Single Cell Data
adata = ad.io.read_h5ad('../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design.h5ad')

# Normalizing to median total counts
sc.pp.normalize_total(adata)
# Logarithmize the data
sc.pp.log1p(adata)

# Limit adata for tests to 5.000
adata = adata[:3000]

print(adata)

/home/boolean/Documents/Uni/Semester_2/Project_CLACELL/.venv/lib/python3.13/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


View of AnnData object with n_obs × n_vars = 3000 × 25825
    obs: 'n_genes', 'Channel', 'n_counts', 'percent_mito', 'scale', 'Group', 'leiden_labels', 'Donor', 'doublet_score', 'pred_dbl', 'anno'
    var: 'featureid', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'mean', 'var', 'hvf_loess', 'hvf_rank'
    uns: 'Channels', 'Groups', 'PCs', 'W_pca_harmony', 'c2gid', 'df_qcplot', 'genome', 'gncells', 'leiden_resolution', 'modality', 'ncells', 'norm_count', 'pca', 'pca_features', 'pca_harmony_knn_distances', 'pca_harmony_knn_indices', 'stdzn_max_value', 'stdzn_mean', 'stdzn_std', 'log1p'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'de_res', 'gmeans', 'gstds', 'means', 'partial_sum'


In [2]:
# Extract raw data as DataFrame
#umi_df = adata.to_df(layer="counts")
umi_df = adata.to_df()
print(umi_df)

featurekey              MIR1302-2HG  AL627309.1  AL627309.3  AL627309.4  \
barcodekey                                                                
BL1_1-AAACCTGAGAGGGATA          0.0         0.0         0.0         0.0   
BL1_1-AAACCTGAGCACCGCT          0.0         0.0         0.0         0.0   
BL1_1-AAACCTGCACGAAACG          0.0         0.0         0.0         0.0   
BL1_1-AAACCTGCAGGGTACA          0.0         0.0         0.0         0.0   
BL1_1-AAACCTGGTTAAAGAC          0.0         0.0         0.0         0.0   
...                             ...         ...         ...         ...   
BL1_1-GCAGTTAAGATGTCGG          0.0         0.0         0.0         0.0   
BL1_1-GCAGTTAAGGTGACCA          0.0         0.0         0.0         0.0   
BL1_1-GCAGTTAAGTTAGCGG          0.0         0.0         0.0         0.0   
BL1_1-GCAGTTACAAATCCGT          0.0         0.0         0.0         0.0   
BL1_1-GCAGTTACAAGCTGAG          0.0         0.0         0.0         0.0   

featurekey              

In [3]:
print(type(umi_df))

<class 'pandas.core.frame.DataFrame'>


In [4]:
import json

with open('../scumi-dev/R/marker_gene/human_pbmc_marker.json', 'r') as file:
    marker_genes = json.load(file)

In [5]:
for marker in marker_genes:
    for gene in marker:
        if gene in umi_df.columns:
            print(gene)

In [11]:
# Not dense
umi = adata.X
print(umi)

<Compressed Sparse Row sparse matrix of dtype 'float32'
	with 1544591 stored elements and shape (7610, 989)>
  Coords	Values
  (0, 2)	1.7377756834030151
  (0, 4)	0.5427432060241699
  (0, 6)	0.8925886154174805
  (0, 8)	0.3077496290206909
  (0, 10)	0.3077496290206909
  (0, 11)	1.7377756834030151
  (0, 13)	0.3077496290206909
  (0, 16)	1.0302627086639404
  (0, 18)	0.3077496290206909
  (0, 19)	1.0302627086639404
  (0, 24)	0.3077496290206909
  (0, 25)	0.3077496290206909
  (0, 26)	0.3077496290206909
  (0, 27)	0.3077496290206909
  (0, 30)	0.3077496290206909
  (0, 31)	2.266817808151245
  (0, 32)	0.5427432060241699
  (0, 35)	3.0006749629974365
  (0, 36)	0.3077496290206909
  (0, 38)	1.3565778732299805
  (0, 42)	3.2101895809173584
  (0, 44)	0.3077496290206909
  (0, 45)	2.469015121459961
  (0, 47)	0.3077496290206909
  (0, 49)	0.3077496290206909
  :	:
  (7609, 757)	2.622593402862549
  (7609, 759)	2.622593402862549
  (7609, 770)	3.2787578105926514
  (7609, 777)	2.622593402862549
  (7609, 789)	2.62259

## python implementation

In [6]:
import json

with open('../scumi-dev/R/marker_gene/human_pbmc_marker.json', 'r') as file:
    marker_genes = json.load(file)

print(marker_genes)

{'CD4+ T cell': ['CD3D+', 'CD3E+', 'CD3G+', 'TRAC+', 'CD4+', 'TCF7+', 'CD27+', 'IL7R+', 'CD8A-', 'CD8B-', 'GNLY-', 'NKG7-', 'CST7-'], 'Cytotoxic T cell': ['CD3D+', 'CD3E+', 'CD3G+', 'TRAC+', 'CD8A+', 'CD8B+', 'GZMK+', 'CCL5+', 'NKG7+', 'CD4-', 'FCER1G-'], 'B cell': ['CD19+', 'MS4A1+', 'CD79A+', 'CD79B+', 'MZB1+', 'IGHD+', 'IGHM+'], 'Natural killer cell': ['NCAM1+', 'NKG7+', 'KLRB1+', 'KLRD1+', 'KLRF1+', 'KLRC1+', 'KLRC2+', 'KLRC3+', 'KLRC4+', 'CD3D-', 'CD3E-', 'CD3G-', 'CD14-', 'FCGR3A+', 'FCGR3B+', 'ITGAL+', 'ITGAM+', 'FCER1G+', 'TRAC-'], 'CD14+ monocyte': ['VCAN+', 'FCN1+', 'S100A8+', 'S100A9+', 'CD14+', 'ITGAL+', 'ITGAM+', 'CSF3R+', 'CSF1R+', 'CX3CR1+', 'FCGR3A-', 'FCGR3B-', 'TYROBP+', 'LYZ+', 'S100A12+', 'CD3D-', 'CD3E-', 'CD3G-', 'TRAC-', 'NKG7-', 'KLRB1-', 'KLRD1-'], 'CD16+ monocyte': ['FCN1+', 'FCGR3A+', 'FCGR3B+', 'ITGAL+', 'ITGAM+', 'CSF3R+', 'CSF1R+', 'CX3CR1+', 'CDKN1C+', 'MS4A7+', 'S100A8-', 'S100A9-', 'S100A12-', 'CD14-', 'CD3D-', 'CD3E-', 'CD3G-', 'TRAC-', 'NKG7-', 'KLRB1

In [7]:
import sys
import os

sys.path.append(os.path.abspath("../python_marker_based_annotation/src"))
from python_marker_based_annotation.model_selection import select_cluster_model

# Run python reimplementation
result_python = select_cluster_model(umi_df.T, dict(marker_genes))

print(result_python)

83 marker genes found in input matrix
SelectionResult(auc_final=B cell                         0.998314
CD14+ monocyte                 0.998158
CD16+ monocyte                 0.977416
CD4+ T cell                    0.974443
Cytotoxic T cell               0.910158
Megakaryocyte                  0.999933
Natural killer cell            0.987791
Plasmacytoid dendritic cell    0.999665
dtype: float64, label_final=barcodekey
BL1_1-AAACCTGAGAGGGATA       Cytotoxic T cell
BL1_1-AAACCTGAGCACCGCT    Natural killer cell
BL1_1-AAACCTGCACGAAACG       Cytotoxic T cell
BL1_1-AAACCTGCAGGGTACA    Natural killer cell
BL1_1-AAACCTGGTTAAAGAC         CD14+ monocyte
                                 ...         
BL1_1-GCAGTTAAGATGTCGG            CD4+ T cell
BL1_1-GCAGTTAAGGTGACCA            CD4+ T cell
BL1_1-GCAGTTAAGTTAGCGG            CD4+ T cell
BL1_1-GCAGTTACAAATCCGT            CD4+ T cell
BL1_1-GCAGTTACAAGCTGAG         CD14+ monocyte
Name: cluster, Length: 3000, dtype: object, params_final={'number_neigh

In [8]:
count = 0
for marker_gene in marker_genes:
    count += len(marker_gene)
print(count)

145


In [9]:
adata.obs['scumi-annotation'] = result_python.label_final
print(adata)

AnnData object with n_obs × n_vars = 3000 × 25825
    obs: 'n_genes', 'Channel', 'n_counts', 'percent_mito', 'scale', 'Group', 'leiden_labels', 'Donor', 'doublet_score', 'pred_dbl', 'anno', 'scumi-annotation'
    var: 'featureid', 'n_cells', 'percent_cells', 'robust', 'highly_variable_features', 'mean', 'var', 'hvf_loess', 'hvf_rank'
    uns: 'Channels', 'Groups', 'PCs', 'W_pca_harmony', 'c2gid', 'df_qcplot', 'genome', 'gncells', 'leiden_resolution', 'modality', 'ncells', 'norm_count', 'pca', 'pca_features', 'pca_harmony_knn_distances', 'pca_harmony_knn_indices', 'stdzn_max_value', 'stdzn_mean', 'stdzn_std', 'log1p'
    obsm: 'X_pca', 'X_pca_harmony', 'X_umap'
    varm: 'de_res', 'gmeans', 'gstds', 'means', 'partial_sum'


In [10]:
# Save annotation
#adata.write(filename=f"{h5ad_file}_annotated.h5ad")
adata.write(filename='../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_annotated.h5ad')

# Save scumi params
#with open(f'{h5ad_file}_params.json', 'w') as file:
with open(f'../data/humancellatlas/5f29c29a-51c6-435c-8ff0-2b2a9d05ebee/BL_standard_design_params.json', 'w') as file:
    file.write(json.dumps(result_python.params_final))